# Step 2: Run the Coding Agent Evals on Amazon Bedrock AgentCore

This notebook evaluates coding agents (Codex, Claude Code, OpenCode, ...) and models on Amazon Bedrock, on tasks made from real pull requests. It explains the scripts in `step2-run-evals/` and runs them in order. To reuse the eval, take the folder `step2-run-evals/` and its [README](step2-run-evals/README.md).

| Step | Folder | Content |
|------|--------|---------|
| 1. Build tasks | [`step1-build-tasks/`](step1-build-tasks/) | Make Harbor tasks from the pull requests of your own repository |
| **2. Run the evals** | `step2-run-evals/` | **This notebook.** Run the tasks with any agent and model, and grade them |

You can start here: `step2-run-evals/datasets/` has four tasks that we made before the workshop.

## 1. What Is an Eval for a Coding Agent?

An eval is a set of examples where you know the correct answer. You run the agent on each example and compare its output to the answer. For a coding agent, an example is a repository, a task in it, and the correct result.

This notebook evaluates two types of agents:

- A **PR review agent** reads a change, reports the defects, and approves or blocks the change.
- A **coding agent** reads an issue and writes a fix.

## 2. The Problem

Public benchmarks test general defects. They do not know the rules of your team or how your services connect, so the examples should come from your own code.

Each example also needs a full, sealed environment. The agent needs the repository at the right commit with its dependencies, and it must not find the answer (the later fix, or the review comments on GitHub).

## 3. The Approach

Your git history is already a labeled dataset:

| Item | Where git keeps it | Use in the eval |
|------|--------------------|-----------------|
| The code before the change | The parent commit | The start state |
| The change | The PR diff | The input for a review agent |
| The review | The PR comments and verdict | The answer for a review agent |
| The fix | The merged patch and its tests | The answer for a coding agent |

For each PR: **roll back** to the parent commit, **seal** the environment, and **compare** the agent's output to the real outcome (the oracle).

| Seal layer | Path to the answer | Control |
|------------|--------------------|---------|
| Git | `git log --all` shows the fix | One commit, no remote, no reflog |
| Network | `gh pr view` shows the review | Allow only the model endpoints |
| Verifier | The agent reads or writes the oracle or the score | Keep the oracle out of the agent's reach |

Step 1 of this module ([`step1-build-tasks.ipynb`](step1-build-tasks.ipynb)) makes tasks this way from your repository.

## 4. The Harbor Task Format

Each example is one [Harbor](https://github.com/harbor-framework/harbor) task directory. The agent sees `instruction.md` and the image that `environment/` builds. Only the verifier sees `tests/` and `solution/`.

```text
tasks/pr-1234-reject/
├── task.toml          # agent and verifier settings, network allowlist
├── instruction.md     # what the agent reads
├── environment/       # Dockerfile: roll back, seal git, install dependencies
├── tests/             # verifier only: test.sh writes reward.json
└── solution/          # the oracle, to test the grader
```

## 5. How the Tasks Run on AgentCore

A **trial** is one agent on one task. The environment `harbor_env.agentcore_env:AgentCoreEnvironment` runs each trial in its own [AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-sessions.html) session: one microVM that starts from the task image and stops after the trial. Only Harbor runs on your computer. You do not need Docker.

```
harbor run ──► AgentCoreEnvironment ──► one AgentCore session for each trial
                                         ├─ the agent works, behind a firewall
                                         ├─ the grader runs
                                         └─ reward.json ──► step2-run-evals/jobs/
```

| Control | What it does |
|---------|--------------|
| Firewall | The agent runs as a non-root user. It can reach only the task's `allowed_hosts`: no GitHub, PyPI, or metadata service. |
| Agent credentials | Short-lived credentials of an agent role that can call only the models that you name. |
| Grading | PR review: in a new microVM. Bug fix: in the agent's microVM, in sealed stages (section 8). |

## 6. Setup

You need Python 3.12, an AWS account where you may create IAM roles, ECR, S3, CodeBuild, and AgentCore runtimes ([permissions](step2-run-evals/README.md#permissions)), and Region `us-east-1`. Setup takes about 15 minutes, most of it the image builds.

In [ ]:
import sys
assert sys.version_info >= (3, 12), f"Python {sys.version.split()[0]}: Harbor needs 3.12 or later. Select a Python 3.12 kernel."
%pip install -q 'harbor==0.26.0' 'boto3>=1.44.7'

Find `step2-run-evals/`, import its notebook helpers (`walkthrough.py`), and check your AWS credentials. To use a named profile, set `AWS_PROFILE` before you start Jupyter, or in this cell.

In [ ]:
import json
import os
import re
from collections import Counter
from pathlib import Path

import boto3

RUN_DIR = Path("step2-run-evals").resolve()
assert RUN_DIR.is_dir(), "Open this notebook from the Coding Assistant folder."
sys.path.insert(0, str(RUN_DIR))
import walkthrough as wt  # the notebook helpers in step2-run-evals/walkthrough.py
from walkthrough import read_json, read_toml

os.environ["AWS_REGION"] = "us-east-1"
# os.environ["AWS_PROFILE"] = "my-profile"

identity = boto3.client("sts", region_name="us-east-1").get_caller_identity()
print("AWS credentials found for:", identity["Arn"].split(":")[-1])

PR_DATASET = RUN_DIR / "datasets/awslabs-mcp-pr-review"
CODING_DATASET = RUN_DIR / "datasets/strands-evals-coding"
REJECT_TASK = PR_DATASET / "tasks/awslabs__mcp__pr4615-reject"
APPROVE_TASK = PR_DATASET / "tasks/awslabs__mcp__pr4121-approve"
ROOT_TASK = CODING_DATASET / "tasks/strands-agents__evals__pr371-fix"
HISTORY_TASK = CODING_DATASET / "tasks/strands-agents__evals__pr338-fix"
PR_TASKS = [REJECT_TASK, APPROVE_TASK]
CODING_TASKS = [ROOT_TASK, HISTORY_TASK]

### Choose the models and bootstrap

`bootstrap` creates the IAM roles, an ECR repository, an S3 bucket, and a CodeBuild project (names start with `evals-hb-`). The agent role may call **only the models in `AGENT_MODELS`**, so list every model that you plan to test. To add one later, run this cell again with the full list.

An `openai.*` ID is called on `bedrock-mantle` (Codex). Any other ID is an inference profile or an on-demand model on `bedrock-runtime`. If a model fails at once with `ThrottlingException`, check its quota in Service Quotas.

In [ ]:
AGENT_MODELS = ["openai.gpt-6-sol", "us.anthropic.claude-sonnet-5-5", "us.amazon.nova-2-lite-v1:0",
                "moonshotai.kimi-k2.5", "zai.glm-5"]

wt.setup("bootstrap", *[a for m in AGENT_MODELS for a in ("--agent-model", m)])

### Build the task images

`prebuild` builds each task image in CodeBuild and deploys an AgentCore runtime for it, in parallel (10 to 15 minutes). The bug-fix images run the repository's test suite at build time, so a failing base fails the build, not a trial.

In [ ]:
wt.setup("prebuild", *map(str, PR_TASKS + CODING_TASKS), tail=25)

### Check the seal from the agent's seat

The task files do not prove that the seal holds at run time. The probe agent (`harbor_env/probe_agent.py`) runs one trial as the agent and tries each path to the answer: the user it runs as, GitHub and PyPI, the metadata service, the hidden folders, the git history, and Bedrock calls with the agent's credentials.

In [ ]:
assert "us.amazon.nova-pro-v1:0" not in AGENT_MODELS, "The probe expects nova-pro to be refused. Set PROBE_OTHER_MODEL to a model that you did not name."

probe_job = wt.harbor_run("probe", [APPROVE_TASK], "-a", "harbor_env.probe_agent:ProbeAgent")
_, trial_dir, _ = next(wt.trials(probe_job))
probe = (trial_dir / "agent" / "probe-host-copy.txt").read_text()
print(re.sub(r"\d{12}", "<account>", probe))

def line(prefix):
    return next((l for l in probe.splitlines() if l.startswith(prefix)), "")

blocked = lambda l: "http=000" in l or "http=403" in l
checks = {
    "runs as the agent user":       "user=agent" in line("PROBE uid="),
    "GitHub blocked":               blocked(line("PROBE github")),
    "PyPI blocked":                 blocked(line("PROBE pypi")),
    "metadata service blocked":     blocked(line("PROBE imds ")) and blocked(line("PROBE imds-via-proxy")),
    "no /tests or /solution":       "No such file" in line("PROBE /tests") and "No such file" in line("PROBE /solution"),
    "one commit, no remote":        "commits=1 remotes=[]" in line("PROBE repo"),
    "agent model allowed":          "ALLOWED" in line("CONVERSE agent-model"),
    "judge model refused":          "REFUSED" in line("CONVERSE judge"),
    "unnamed model refused":        "REFUSED" in line("CONVERSE other-model"),
}
for name, ok in checks.items():
    print(f"{'OK ' if ok else 'ERR'} {name}")
assert all(checks.values()), "A probe failed. Read the output above before you run an agent."

## 7. PR Review Eval

You practice on real cases from an open-source project: two pull requests of [awslabs/mcp](https://github.com/awslabs/mcp) that its reviewers blocked or approved, made into tasks with [eval-harvest](https://github.com/aws-samples/sample-code-eval-harvester):

| | Reject | Approve |
|---|---|---|
| PR | [#4615](https://github.com/awslabs/mcp/pull/4615): read-only SQL guard | [#4121](https://github.com/awslabs/mcp/pull/4121): EKS TLS verification |
| Expected verdict | `block` | `approve` |
| Oracle | 1 high, 2 medium, 3 low findings | no findings |

An agent that blocks every PR passes the reject task but fails the approve task, so you need both.

### 7.1 Check the task files

Shared verifier files live in `datasets/_shared/` and are copied into each task by `sync.py`.

In [ ]:
for task_dir in PR_TASKS:
    wt.show_tree(task_dir)
    print()
wt.check_shared_files()

In `task.toml`, the agent may reach only Bedrock, and the verifier runs in a separate microVM, so the oracle is never in the agent's microVM.

In [ ]:
for task_dir in PR_TASKS:
    cfg = read_toml(task_dir / "task.toml")
    env, verifier = cfg["environment"], cfg["verifier"]
    print(f"{task_dir.name}")
    print(f"  agent network:    {env['network_mode']} {env.get('allowed_hosts', [])}")
    print(f"  verifier:         {verifier['environment_mode']}")
    print(f"  metadata.harvest: {cfg['metadata']['harvest']}")
    print()

    assert env["network_mode"] == "allowlist", "The agent network must be an allowlist."
    assert all("bedrock" in host for host in env["allowed_hosts"]), "Allow only Bedrock endpoints."
    assert not any("github" in host for host in env["allowed_hosts"]), "The agent must not reach GitHub."
    assert verifier["environment_mode"] == "separate", "The verifier must be in a separate environment."
print("task.toml checks passed.")

`instruction.md` is the same for every task and does not mention the PR. The agent writes its findings to `/logs/artifacts/findings.json`. A missing file counts as no review.

In [ ]:
instructions = {t.name: (t / "instruction.md").read_text() for t in PR_TASKS}
assert len(set(instructions.values())) == 1, "The instructions are not the same for all tasks."
print(instructions[REJECT_TASK.name])

In `tests/oracle.json`, each finding is a defect with a severity, and its location touches a line that the change adds. Only `high` findings block.

In [ ]:
def added_lines(patch: str) -> dict[str, set[int]]:
    # The line numbers that a unified diff adds, for each file.
    added, path, line_no = {}, None, None
    for line in patch.splitlines():
        if line.startswith("+++ "):
            path = line[6:] if line.startswith("+++ b/") else None
            added.setdefault(path, set())
        elif m := re.match(r"@@ -\d+(?:,\d+)? \+(\d+)(?:,\d+)? @@", line):
            line_no = int(m.group(1))
        elif path and line_no is not None and not line.startswith(("---", "\\")):
            if line.startswith("+"):
                added[path].add(line_no)
                line_no += 1
            elif line.startswith(" "):
                line_no += 1
    return added

for task_dir in PR_TASKS:
    oracle = read_json(task_dir / "tests" / "oracle.json")
    expected = read_toml(task_dir / "task.toml")["metadata"]["harvest"]["expected_verdict"]
    changed = added_lines((task_dir / "environment" / "change.patch").read_text())
    print(f"{task_dir.name}: expected_verdict={oracle['expected_verdict']}")

    assert oracle["expected_verdict"] == expected == read_json(task_dir / "tests" / "verdict.json")["expected_verdict"]
    for f in oracle["findings"]:
        assert f["severity"] in ("low", "medium", "high")
        for loc in f["locations"]:
            span = set(range(loc["line_start"], loc["line_end"] + 1))
            assert span & changed.get(loc["path"], set()), f"Finding {f['id']} does not touch an added line."
        flag = " (reference only)" if f["reference_only"] else ""
        spans = ", ".join(f"{l['line_start']}-{l['line_end']}" for l in f["locations"])
        print(f"  {f['severity']:<6} {f['locations'][0]['path'].split('/')[-1]}:{spans}{flag}")
        print(f"         {f['statement'][:110]}...")

    blocking = [f for f in oracle["findings"] if f["severity"] == "high"]
    assert (expected == "block") == bool(blocking), "Only a task with a high finding can expect `block`."
    print()
print("oracle.json checks passed.")

The leakage scan makes sure that the agent-visible files do not contain the PR number, the reviewer names, or the review text. It first checks itself with a canary.

In [ ]:
def pr_needles(pr: int) -> list[str]:
    # The PR number, the reviewer names, and the first 60 characters of each review comment.
    cand = read_json(PR_DATASET / "candidates" / f"pr-{pr}.json")
    needles = [f"#{pr}", f"pull/{pr}"]
    needles += sorted({v["author"] for v in cand["review_verdicts"]})
    needles += [c["body"][:60] for c in cand["comments"] if len(c["body"]) >= 60]
    return needles

for task_dir, pr in [(REJECT_TASK, 4615), (APPROVE_TASK, 4121)]:
    needles = pr_needles(pr)
    hits = wt.leakage_scan(task_dir, needles)
    print(f"{task_dir.name}: {len(needles)} needles, " + ("no leaks." if not hits else f"leaks: {hits}"))

### 7.2 Metrics

`tests/score.py` matches each agent finding to the oracle by location, then an LLM judge decides if they describe the same defect.

| Signal | Question |
|--------|----------|
| `passed` | **The result.** Correct verdict and, on a reject task, every high finding found |
| `submitted` | Did the agent write a findings file? |
| `coverage_required`, `coverage_all` | What fraction of the high findings, and of all findings, did the agent find? |
| `precision_adjudicated` | What fraction of the agent's findings match the oracle? Undefined with no findings |
| `uncredited` | Findings that match nothing. They are not counted as errors, because reviewers miss defects too |

### 7.3 Test the grader

The `oracle` agent submits the oracle and must pass. The `nop` agent does nothing and must fail.

In [ ]:
oracle = wt.trial_rewards(wt.harbor_run("review-oracle", PR_TASKS, "-a", "oracle"))
nop = wt.trial_rewards(wt.harbor_run("review-nop", PR_TASKS, "-a", "nop"))

for task in sorted(oracle):
    print(f"{task}")
    print(f"  oracle: passed={oracle[task].get('passed')} coverage_all={oracle[task].get('coverage_all')} uncredited={oracle[task].get('uncredited')}")
    print(f"  nop:    passed={nop[task].get('passed')} submitted={nop[task].get('submitted')}")
    assert oracle[task].get("passed") == 1.0, "The grader did not pass the oracle."
    assert nop[task].get("passed") == 0.0, "The grader passed an agent that did nothing."
assert oracle[REJECT_TASK.name].get("precision_adjudicated") == 1.0
assert "precision_adjudicated" not in oracle[APPROVE_TASK.name], "Precision must be undefined when there are no findings."
print("The oracle passes and the no-op agent fails.")

### 7.4 Run Codex

The flags point Codex at Bedrock and turn off web search. Do not pass AWS keys with `--ae`: the environment gives the agent its own credentials. `N_ATTEMPTS` is the number of trials for each task.

In [ ]:
print((RUN_DIR / "codex-bedrock.toml").read_text())

CODEX = ["-a", "codex", "-m", "openai.gpt-6-sol", "--skip-model-parsing-and-provider-mapping",
         "--ak", "config=codex-bedrock.toml", "--ak", "disable_web_search=true", "--ae", "AWS_REGION=us-east-1"]

In [ ]:
N_ATTEMPTS = 1

review_codex = wt.harbor_run("review-codex", PR_TASKS, *CODEX, attempts=N_ATTEMPTS)
wt.summary(review_codex)

### 7.5 Results

`grading` must be `separate`. This cell adds the dataset metrics: accuracy, block recall and precision, and the false approve and false block rates.

In [ ]:
def review_rows(job_dir: Path) -> list[dict]:
    rows = []
    for task, trial_dir, result in wt.trials(job_dir):
        if result.get("exception_info"):
            print(f"INVALID {task}: {result['exception_info'].get('exception_type')}")
            continue
        reward = read_json(trial_dir / "verifier" / "reward.json")
        findings = read_json(trial_dir / "artifacts/logs/artifacts/findings.json", [])
        harvest = read_toml(PR_DATASET / "tasks" / task / "task.toml")["metadata"]["harvest"]
        rows.append({"task": task, "risk": harvest["change_risk_classified"], "findings": findings,
                     "expected": "block" if reward["expected_blocks"] else "approve",
                     "predicted": "block" if reward.get("agent_blocks") else "approve", **reward})
    return rows

def review_metrics(rows: list[dict]) -> dict:
    should_block = [r for r in rows if r["expected"] == "block"]
    did_block = [r for r in rows if r["predicted"] == "block"]
    clean = [r for r in rows if r["expected"] == "approve"]
    rate = lambda num, den: round(len(num) / len(den), 3) if den else None
    return {
        "tasks": len(rows),
        "accuracy": rate([r for r in rows if r["passed"]], rows),
        "block_recall": rate([r for r in should_block if r["predicted"] == "block"], should_block),
        "block_precision": rate([r for r in did_block if r["expected"] == "block"], did_block),
        "false_approve_rate": rate([r for r in should_block if r["predicted"] == "approve"], should_block),
        "false_block_rate": rate([r for r in clean if r["predicted"] == "block"], clean),
    }

rows = review_rows(review_codex)
for r in rows:
    print(f"{'PASS' if r['passed'] else 'FAIL'} {r['task']}")
    if not r.get("submitted", 1):
        print("     no findings file: read the agent's log in Harbor view")
    print(f"     verdict: expected {r['expected']}, predicted {r['predicted']}")
    print(f"     coverage_required={r['coverage_required']} coverage_all={r['coverage_all']} uncredited={r['uncredited']:.0f}")
    for f in r["findings"]:
        print(f"     - [{f.get('severity')}] {str(f.get('path', '')).split('/')[-1]}:{f.get('line')} {str(f.get('statement', ''))[:100]}")
print()
print(json.dumps(review_metrics(rows), indent=2))

To find why a trial failed, open it in Harbor view (section 10). Common causes:

| Cause | What to do |
|-------|------------|
| The agent missed a defect that it could find | Change its prompt or tools, and run again |
| `submitted` is 0: the review is in the reply, not the file | Read the log; change the prompt or the harness |
| The datapoint is wrong (a later commit fixed the "approved" code) | Fix the oracle or pick another PR |
| A correct finding is just outside the oracle span | Widen the span in `oracle.json` |

## 8. Bug-Fix Eval

Two real bugs from [strands-agents/evals](https://github.com/strands-agents/evals). The agent gets the issue report and the repository rolled back to before the fix, and writes the fix. The grading follows [Senior SWE-Bench](https://senior-swe-bench.snorkel.ai/blog/2026-06-16-how-it-works): a fix must be correct *and* one that a maintainer would merge.

In [ ]:
for task_dir in CODING_TASKS:
    meta = read_toml(task_dir / "task.toml")["metadata"]
    origin, verifier, story = meta["origin"], meta["verifier"], meta["narrative"]
    print(f"{task_dir.name}")
    print(f"  origin:   {origin['repo']} PR {origin['pr_numbers']}, issue {origin['issue_numbers']}, base {origin['base_commit'][:9]}")
    print(f"  tests:    {verifier['fail_to_pass']} f2p, {verifier['pass_to_pass']} p2p, in {verifier['hidden_tests']}")
    print(f"  cause:    {' '.join(story['description'].split())}")
    print(f"  fix:      {' '.join(story['solution'].split())}")
    print()

### 8.1 The reward stack

| Mechanism | Checks | Gate |
|-----------|--------|------|
| Hidden tests | Fail-to-pass (the bug is fixed), pass-to-pass and the base suite (nothing broke) | All pass |
| Validation agent | Tests written for the shape of the agent's change, from a spec | All stories pass |
| Rubric judge | Task criteria that tests cannot check | > 0.5 |
| Taste judge | Yes/no checks against the merged fix and the codebase | ≥ 3 of 5 in each group |
| Patch bloat | Agent patch size / merged patch size | < 2× |

`correct` is 1 if the tests and the validation pass. `reward` is 1 if every gate passes.

### 8.2 Check the task files

In [ ]:
for task_dir in CODING_TASKS:
    wt.show_tree(task_dir)
    print()
wt.check_shared_files()

In `task.toml`, the verifier runs in the agent's microVM, because it must run the agent's code. Its phase may reach only the Bedrock runtime endpoint, with the grader role.

In [ ]:
for task_dir in CODING_TASKS:
    cfg = read_toml(task_dir / "task.toml")
    env, verifier = cfg["environment"], cfg["verifier"]
    print(f"{task_dir.name}")
    print(f"  agent network: {env['network_mode']} {env.get('allowed_hosts', [])}")
    print(f"  verifier:      {verifier.get('environment_mode', 'shared')}, network {verifier['network_mode']} {verifier['allowed_hosts']}")
    print(f"  timeouts:      agent {cfg['agent']['timeout_sec']:.0f} s, verifier {verifier['timeout_sec']:.0f} s")
    print()

    assert env["network_mode"] == "allowlist", "The agent network must be an allowlist."
    assert all("bedrock" in host for host in env["allowed_hosts"]), "Allow only Bedrock endpoints."
    assert verifier.get("environment_mode", "shared") == "shared", "The verifier must run the agent's code."
    assert verifier["allowed_hosts"] == ["bedrock-runtime.us-east-1.amazonaws.com"], "The verifier may reach only Bedrock."
print("task.toml checks passed.")

The instruction is an issue report. It gives the symptom and the expected behavior, not the file or the cause.

In [ ]:
print((ROOT_TASK / "instruction.md").read_text())

In [ ]:
def coding_needles(task_dir: Path) -> list[str]:
    # The PR and issue numbers, the merge commit, the hidden test file, and the long lines that the fix adds.
    meta = read_toml(task_dir / "task.toml")["metadata"]
    origin = meta["origin"]
    needles = [f"#{n}" for n in origin["pr_numbers"] + origin["issue_numbers"]]
    needles += [f"pull/{n}" for n in origin["pr_numbers"]] + [origin["merge_commit"][:12]]
    needles += [Path(meta["verifier"]["hidden_tests"]).stem]
    patch = (task_dir / "solution" / "oracle.patch").read_text()
    needles += [line[1:].strip() for line in patch.splitlines()
                if line.startswith("+") and not line.startswith("+++") and len(line.strip()) >= 40]
    return needles

for task_dir in CODING_TASKS:
    needles = coding_needles(task_dir)
    hits = wt.leakage_scan(task_dir, needles)
    print(f"{task_dir.name}: {len(needles)} needles, " + ("no leaks." if not hits else f"leaks: {hits}"))

### 8.3 Staged grading

The grader runs the agent's code (its fix and the tests). If that code could write `reward.json`, the agent could score itself. So `test.sh` runs in two stages:

1. All agent processes are killed, and the credentials switch to the grader role.
2. `test.sh --stage untrusted` runs the tests as the agent's user. Its output is evidence only.
3. Root copies that output (never a reward file), and `test.sh --stage trusted` runs the judges and the aggregation as a separate `grader` user. Only this stage's `reward.json` counts.

The `grading` column shows how a trial was graded: `staged` (as above), `separate` (PR review), `unsealed` (the task's `test.sh` has no `--stage`, so the reward is not isolated), or `refused`. To use staged grading in your own tasks, copy the sample tasks' `tests/test.sh`.

### 8.4 Test the reward stack

The oracle (the merged fix) must be a tasteful solve, and `nop` must fail with `pass_to_pass` at 1. About 5 minutes.

In [ ]:
SHOW = ["reward", "correct", "fail_to_pass", "pass_to_pass", "existing_suite", "validation_score",
        "rubric_score", "relative_taste", "practice_alignment", "patch_bloat"]

oracle_job = wt.harbor_run("coding-oracle", [ROOT_TASK], "-a", "oracle")
nop_job = wt.harbor_run("coding-nop", [ROOT_TASK], "-a", "nop")
oracle = wt.trial_rewards(oracle_job)[ROOT_TASK.name]
nop = wt.trial_rewards(nop_job)[ROOT_TASK.name]
print("oracle:", {k: oracle.get(k) for k in SHOW})
print("nop:   ", {k: nop.get(k) for k in SHOW})
for job in (oracle_job, nop_job):
    _, trial_dir, _ = next(wt.trials(job))
    print(f"{job.name}: grading {wt.grading_mode(trial_dir)}")
    assert wt.grading_mode(trial_dir) == "staged", "The trial was not graded in stages."

assert oracle.get("reward") == 1.0, "The oracle is not a tasteful solve. See verifier/reward_details.json."
assert nop.get("correct") == 0.0 and nop.get("fail_to_pass") == 0.0, "The no-op agent passes a correctness check."
assert nop.get("pass_to_pass") == 1.0 and nop.get("existing_suite") == 1.0, "A test fails on the base."
print("The oracle is a tasteful solve, and the no-op agent fails.")

### 8.5 Run Codex

This takes about 10 minutes. The agent runs the tests many times, then the validation agent and the judges run.

In [ ]:
coding_codex = wt.harbor_run("coding-codex", CODING_TASKS, *CODEX, attempts=N_ATTEMPTS)
wt.summary(coding_codex)

In [ ]:
def coding_rows(job_dir: Path) -> list[dict]:
    rows = []
    for task, trial_dir, result in wt.trials(job_dir):
        details = read_json(trial_dir / "verifier" / "reward_details.json", {"invalid": ["no verifier output"]})
        rows.append({
            "task": task, "grading": wt.grading_mode(trial_dir), "invalid": details.get("invalid"),
            "error": (result.get("exception_info") or {}).get("exception_type"),
            "checks": details.get("checks", {}), **details.get("flat", {}),
            "failed_tests": details.get("failed_tests", []),
            "stories": {k: f"{v['passed']}/{v['total']}" for k, v in details.get("validation", {}).get("stories", {}).items()},
            "taste": {k: v["passed"] for k, v in details.get("taste", {}).get("dimensions", {}).items()},
        })
    return rows

rows = coding_rows(coding_codex)
for r in rows:
    if r["error"] or r["invalid"]:
        print(f"INVALID {r['task']}: {r['error'] or '; '.join(r['invalid'])}")
        continue
    status = "TASTEFUL SOLVE" if r["reward"] == 1.0 else ("CORRECT, NOT TASTEFUL" if r["correct"] == 1.0 else "FAIL")
    print(f"{status}  {r['task']}  (grading: {r['grading']})")
    for gate, ok in r["checks"].items():
        print(f"    {'pass' if ok else 'FAIL'}  {gate}")
    print(f"    tests:      f2p={r['fail_to_pass']} p2p={r['pass_to_pass']} existing={r['existing_suite']}")
    print(f"    validation: {r['stories']}")
    print(f"    rubric={r['rubric_score']} relative_taste={r['relative_taste']} practice={r['practice_alignment']} bloat={r['patch_bloat']}")
    print(f"    agent tests: added={r['agent_tests_added']:.0f} reproduce_bug={r['agent_tests_reproduce_bug']:.0f}")
    for line in r["failed_tests"][:10]:
        print(f"    failed: {line}")

scored = [r for r in rows if not (r["error"] or r["invalid"])]
if scored:
    print(f"\nCorrect:         {sum(r['correct'] for r in scored):.0f}/{len(scored)} trials")
    print(f"Tasteful solves: {sum(r['reward'] for r in scored):.0f}/{len(scored)} trials")

Each trial folder has the agent's patch (`verifier/agent.patch`), the failed tests (`verifier/failed_tests.txt`), and the reason for each judge decision (`verifier/rubric.json`, `taste.json`). Common causes:

| Cause | What to do |
|-------|------------|
| The fix is wrong or incomplete (`fail_to_pass` < 1) | Change the agent's prompt or procedure |
| The fix breaks something (`pass_to_pass` < 1) | Same; the agent did not run the full suite |
| A hidden test checks one design, not the behavior | Rewrite the test, then rerun oracle and `nop` |
| Correct but not tasteful | Read `taste.json`; if you disagree, change the judge |

## 9. Change the Harness and the Model

An agent is a **harness** (`-a`: the loop, the tools) and a **model** (`-m`). Change one at a time to see which one makes the difference:

| Run | Harness | Model | Shows |
|-----|---------|-------|-------|
| Base | Claude Code | Claude Sonnet 5.5 | |
| Change the harness | **OpenCode** | Claude Sonnet 5.5 | the harness effect |
| Change the model | OpenCode | **Nova 2 Lite**, **Kimi K2.5**, **GLM 5** | the model effect |

Each harness needs flags to work in the sandbox: its Bedrock endpoint, its side model set to the same model, and update checks off.

In [ ]:
def claude_code(model: str) -> list[str]:
    return ["-a", "claude-code", "-m", model, "--skip-model-parsing-and-provider-mapping",
            "--ae", "CLAUDE_CODE_USE_BEDROCK=1",
            "--ae", f"ANTHROPIC_DEFAULT_HAIKU_MODEL={model}", "--ae", f"ANTHROPIC_SMALL_FAST_MODEL={model}",
            "--ae", "CLAUDE_CODE_DISABLE_NONESSENTIAL_TRAFFIC=1", "--ae", "DISABLE_AUTOUPDATER=1",
            "--ae", "AWS_REGION=us-east-1"]

def opencode(model: str) -> list[str]:
    config = {"small_model": f"amazon-bedrock/{model}", "autoupdate": False, "share": "disabled"}
    return ["-a", "opencode", "-m", f"amazon-bedrock/{model}", "--ae", "AWS_REGION=us-east-1",
            "--ae", "OPENCODE_DISABLE_MODELS_FETCH=1", "--ae", "OPENCODE_DISABLE_AUTOUPDATE=1",
            "--ae", "OPENCODE_DISABLE_LSP_DOWNLOAD=1", "--ae", "OPENCODE_DISABLE_DEFAULT_PLUGINS=1",
            "--ae", "OPENCODE_DISABLE_CLAUDE_CODE=1", "--ak", f"opencode_config={json.dumps(config)}"]

SONNET = "us.anthropic.claude-sonnet-5-5"
RUNS = {
    "claude-code-sonnet": claude_code(SONNET),                # base
    "opencode-sonnet": opencode(SONNET),                      # change the harness
    "opencode-nova": opencode("us.amazon.nova-2-lite-v1:0"),  # change the model
    "opencode-kimi": opencode("moonshotai.kimi-k2.5"),
    "opencode-glm": opencode("zai.glm-5"),
}
for name, args in RUNS.items():
    model = args[args.index("-m") + 1].removeprefix("amazon-bedrock/")
    assert model in AGENT_MODELS, f"{name}: {model} is not in AGENT_MODELS. Add it and run bootstrap again."
    print(name, " ".join(args[:4]))

The comparison runs on the two PR review tasks (ten trials). It is off by default. Set `COMPARE_TASKS = CODING_TASKS` to compare on the bug fixes.

In [ ]:
RUN_COMPARE = False
COMPARE_TASKS = PR_TASKS

compare_jobs = {}
if RUN_COMPARE:
    for name, args in RUNS.items():
        compare_jobs[name] = wt.harbor_run(f"compare-{name}", COMPARE_TASKS, *args, attempts=N_ATTEMPTS)
    wt.summary(*compare_jobs.values())
else:
    print("Skipped. Set RUN_COMPARE = True to run it.")

### One trial is not enough

With two tasks and one trial each, a difference can be chance. `attempts=3, concurrent=3` runs 3 trials of each task, 3 at a time. More tasks help more than more trials, and step 1 ([`step1-build-tasks.ipynb`](step1-build-tasks.ipynb)) makes tasks from your own repository.

To test another model, add it to `AGENT_MODELS` and run bootstrap again. To add another Harbor agent, install it in `step2-run-evals/harbor_env/image/layer.Dockerfile` and allow its endpoint in each task's `allowed_hosts`.

In [ ]:
RUN_REPEATS = False
REPEATS = 3

if RUN_REPEATS:
    repeat_jobs = [wt.harbor_run(f"repeat-{name}", COMPARE_TASKS, *args, attempts=REPEATS, concurrent=3)
                   for name, args in RUNS.items()]
    wt.summary(*repeat_jobs)
    for job in repeat_jobs:
        passed = [r.get("passed") or r.get("reward") or 0 for r in [read_json(d / "verifier/reward.json", {}) for _, d, _ in wt.trials(job)]]
        print(f"{job.name}: passed {sum(passed):.0f} of {len(passed)} trials")
else:
    print("Skipped. Set RUN_REPEATS = True to run it.")

## 10. Look at the Results in Harbor View

`harbor view` shows each trial: the reward, every command and message of the agent, and the files it changed. It runs on `127.0.0.1` only.

In [ ]:
viewer, viewer_url = wt.start_harbor_view()

In [ ]:
wt.stop_harbor_view(viewer)

## 11. Clean Up

`teardown` deletes the AWS resources of the prefix (about 4 minutes), and `resources` must then print empty lists. Teardown keeps the local `step2-run-evals/jobs/`. It holds agent and judge transcripts, so review it before you share it.

In [ ]:
RUN_TEARDOWN = False  # set to True when you are done with this notebook

if RUN_TEARDOWN:
    wt.setup("teardown", tail=10)
    wt.setup("resources", tail=30)
else:
    print("Skipped. Set RUN_TEARDOWN = True to delete the AWS resources.")

## Summary

- Build examples from your PR history: roll back, seal, compare to the real outcome.
- Check the seal from the agent's seat, and run the oracle and `nop` before any agent.
- Grade so that the agent's code cannot write its own score.
- Change one part of the agent at a time, and run several trials before you compare.

**Next:** make tasks from your own repository with step 1 ([`step1-build-tasks.ipynb`](step1-build-tasks.ipynb)), and run them here. See [Bring your own tasks](step2-run-evals/README.md#bring-your-own-tasks) for what a task needs.